# Advanced Retrieval with the Cat Health PDF

Session 1 used a dense vector retriever:

```text
question -> embed -> nearest chunks -> answer
```

This notebook keeps the same cat-health PDF and compares dense vector search, BM25, parent-child retrieval, hybrid retrieval, Cohere reranking, and multi-query retrieval.

> This is a retrieval exercise, not veterinary guidance. Answer only from retrieved context. Recommend a veterinarian for diagnosis, treatment, medication, or urgent-care decisions.


## Learning Outcomes

By the end of this session, you will be able to:

- Explain the different failure modes of dense and BM25 retrieval.
- Fuse independent ranked lists with reciprocal rank fusion (RRF).
- Increase recall with multiple generated search queries.
- Search focused child chunks while returning useful parent-page context.
- Use Cohere Rerank as a second-stage ranker.
- Compare retrieval systems with reviewed cases, visible evidence, metrics, and latency.


## Build Order

Build and compare the following layers:

1. Naive dense RAG: in-memory Qdrant, OpenAI embeddings, nearest child chunks, and a grounded answer.
2. BM25: sparse lexical retrieval over the same chunks.
3. Parent-child retrieval: search precise child chunks and return their parent PDF pages.
4. Hybrid retrieval with Cohere reranking: fuse dense and BM25 candidates with reciprocal rank fusion (RRF), recover parent pages, then rerank them.
5. Multi-query retrieval: generate alternate searches before the hybrid retrieve-then-rerank pipeline.

Dense retrieval plus BM25 is **hybrid retrieval** (or hybrid search). RRF is an **ensemble** method that combines their ranked lists. Adding Cohere reranking creates a **two-stage hybrid retrieve-then-rerank pipeline**.

Extra stages add latency, cost, and sometimes noise. Evaluate each added stage against those trade-offs.


## Task 1: Setup

Install the project environment from the session folder with `uv sync`, then run this notebook from that same folder. You need `OPENAI_API_KEY` and `COHERE_API_KEY` available when the notebook prompts for them.


In [1]:
from __future__ import annotations

import os
import re
from dataclasses import replace
from getpass import getpass
from pathlib import Path
from typing import Iterable, Sequence

import pandas as pd
from langchain_cohere import CohereRerank
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi

from lib import (
    AnswerOutput,
    EvalCase,
    EvalItem,
    RetrievedDocument,
    answer_similarity_scorer,
    compare_eval_reports,
    compare_reports,
    faithfulness_scorer,
    make_openai_faithfulness_judge,
    run_eval,
    run_retrieval_eval,
)


/var/folders/fw/kwy0s5tj0l39fzpsqntd04nm0000gn/T/ipykernel_60268/2795770623.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass("Cohere API Key: ")

CHAT_MODEL = os.environ.get("AIM_CHAT_MODEL", "gpt-5.4-mini")
EVAL_MODEL = os.environ.get("AIM_EVAL_MODEL", CHAT_MODEL)
EMBEDDING_MODEL = os.environ.get("AIM_EMBEDDING_MODEL", "text-embedding-3-small")
RERANK_MODEL = os.environ.get("AIM_RERANK_MODEL", "rerank-v4.0-fast")

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
answer_model = ChatOpenAI(model=CHAT_MODEL, temperature=0)
evaluation_model = ChatOpenAI(model=EVAL_MODEL, temperature=0)
query_model = ChatOpenAI(model=CHAT_MODEL, temperature=0)

print(f"Chat model: {CHAT_MODEL}")
print(f"Evaluation model: {EVAL_MODEL}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Cohere rerank model: {RERANK_MODEL}")


Chat model: gpt-5.4-mini
Evaluation model: gpt-5.4-mini
Embedding model: text-embedding-3-small
Cohere rerank model: rerank-v4.0-fast


## Task 2: Load and Chunk the Cat PDF for Naive RAG

Load the bundled PDF as page-level documents, then split each page into focused chunks for the dense RAG baseline. Preserve source and page metadata on every chunk.


In [3]:
pdf_path = Path("data/cat_health_guidelines.pdf")
if not pdf_path.exists():
    raise FileNotFoundError(f"Expected the cat PDF at {pdf_path.resolve()}")

pages = [page for page in PyPDFLoader(str(pdf_path)).load() if page.page_content.strip()]
for page_number, page in enumerate(pages, start=1):
    page.metadata.update(
        {
            "source": pdf_path.name,
            "page": page_number,
            "page_id": f"page-{page_number:02d}",
            "document_type": "cat_health_guideline",
        }
    )

print(f"Loaded {len(pages)} text-containing PDF pages.")
print(pages[0].metadata)


Loaded 22 text-containing PDF pages.
{'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 1, 'page_label': '1', 'page_id': 'page-01', 'document_type': 'cat_health_guideline'}


In [6]:
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    add_start_index=True,
)
children = child_splitter.split_documents(pages)
for index, child in enumerate(children):
    child.metadata["chunk_id"] = f"child-{index:03d}"

print(f"Created {len(children)} child chunks from {len(pages)} parent pages.")
print(children[0].metadata)
print(children[0].page_content[:500])


Created 159 child chunks from 22 parent pages.
{'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 1, 'page_label': '1', 'page_id': 'page-01', 'document_type': 'cat_health_guideline', 'start_index': 0, 'chunk_id': 'child-000'}
VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are publishe


### ❓ Question 1: Traceability

Why should every child chunk retain its source file and page metadata?

### Answer:

Every child chunk should retain its source file and page metadata so we know exactly where the information came from. This makes it easier to trace and verify the retrieved content, provide citations, and debug retrieval issues. In the parent-child retriever we will be exploring later on, the metadata also helps map the retrieved child chunk back to its parent document so the LLM gets more complete context. It can also be useful for grouping or deduplicating chunks from the same source.


### Shared Result Representation

All retrievers return the same RetrievedDocument type. Stable chunk and page IDs make the evidence inspectable and keep later comparisons fair.


In [7]:
def as_retrieved_document(document: Document, score: float | None = None) -> RetrievedDocument:
    return RetrievedDocument(
        id=document.metadata["chunk_id"],
        text=document.page_content,
        score=float(score) if score is not None else None,
        evidence_ids=(document.metadata["page_id"],),
        metadata=dict(document.metadata),
    )


def print_results(results: Sequence[RetrievedDocument], text_limit: int = 260) -> None:
    for rank, result in enumerate(results, start=1):
        page = result.metadata.get("page", "?")
        score = "n/a" if result.score is None else f"{result.score:.4f}"
        print(f"#{rank} | {result.id} | page={page} | score={score}")
        print(result.text[:text_limit].replace("\n", " "))
        print()


## Task 3: Naive Dense Vector RAG with In-Memory Qdrant

Session 1 baseline:

question -> OpenAI embedding -> Qdrant nearest-neighbor search -> answer

Create an in-memory Qdrant collection from the child chunks. Retrieve the nearest chunks, then pass only those chunks to the answer model. Use this **naive RAG baseline** for later comparisons.


In [8]:
# Qdrant stays in memory for this notebook. It disappears when the kernel stops.
vector_store = QdrantVectorStore.from_documents(
    documents=children,
    embedding=embeddings,
    location=":memory:",
    collection_name="cat_health_naive_dense_rag",
)

FIRST_STAGE_K = 8


def dense_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    matches = vector_store.similarity_search_with_score(question, k=k)
    return [as_retrieved_document(document, score) for document, score in matches]


dense_preview = dense_retrieve("What should a senior cat wellness visit cover?", k=4)
print_results(dense_preview)


#1 | child-034 | page=7 | score=0.6789
disease. Mature Adult and Senior Cats The medical history and examination of mature adult and senior cats will be focused on early detection of disease. Adult and senior cats are often diagnosed with comorbidities. Speci ﬁc questions regarding changes in appet

#2 | child-021 | page=6 | score=0.6741
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at the veterinarian ’s discretion. The guidelines are intended to be a starting point from which individualized care re

#3 | child-060 | page=10 | score=0.6188
study, three 10- to 15-minute exercise sessions per day led to a loss of approximately 1% of body weight in 1 month with no food intake restrictions. 66 Senior Cats Senior cats exhibiting new or unusual behavior should be evaluated for medical conditions. 12 C

#4 | child-036 | page=8 | score=0.6184
Detecting signs of pain or anxiety and evaluation of qual

### Naive RAG Answer

Retrieval quality and answer quality are separate concerns. Pass the nearest dense chunks to the answer model. The same grounded-answer function will later receive context from the other retrievers.


In [9]:
answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer only from the provided cat-health guideline context. "
            "Do not diagnose, prescribe, or make an urgent-care decision. "
            "If the context is insufficient, say so. End with a short Sources line.",
        ),
        ("human", "Question: {question}\n\nContext:\n{context}"),
    ]
)


def format_context(documents: Sequence[RetrievedDocument]) -> str:
    blocks = []
    for document in documents:
        page = document.metadata.get("page", "unknown")
        blocks.append(f"[source={document.id}; page={page}]\n{document.text}")
    return "\n\n".join(blocks)


def answer_with_retriever(
    question: str,
    retriever,
    k: int = 4,
) -> AnswerOutput:
    documents = retriever(question, k=k)
    response = answer_model.invoke(
        answer_prompt.format_messages(question=question, context=format_context(documents))
    )
    return AnswerOutput(answer=str(response.content), documents=tuple(documents))


naive_result = answer_with_retriever(
    "What should an owner discuss at a senior cat wellness visit?",
    retriever=dense_retrieve,
)
print(naive_result.answer)
print("\nNaive dense evidence:")
print_results(naive_result.documents, text_limit=200)


At a senior cat wellness visit, the owner should discuss any changes in:

- Appetite or eating habits
- Urination and drinking, including polyuria/polydipsia
- Vomiting, hairballs, or diarrhea
- Litter box usage
- Nighttime activity or vocalization
- Normal habits, activity level, mobility, or behavior
- Possible signs related to pain, reduced vision/hearing, or cognitive changes

It’s also appropriate to review the cat’s diet and body weight/body condition as part of the visit.

Sources: child-034 p.7; child-021 p.6; child-060 p.10; child-028 p.6

Naive dense evidence:
#1 | child-034 | page=7 | score=0.6958
disease. Mature Adult and Senior Cats The medical history and examination of mature adult and senior cats will be focused on early detection of disease. Adult and senior cats are often diagnosed with 

#2 | child-021 | page=6 | score=0.6693
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at

## Task 4: BM25 Sparse Retrieval

BM25 ranks the same child chunks with lexical term matches rather than embeddings. It is useful for abbreviations, age ranges, named conditions, and phrases that dense similarity may blur.

Keep the corpus, chunks, and retrieval depth fixed. The retriever is the only thing that changes.


In [10]:
def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())


bm25 = BM25Okapi([tokenize(child.page_content) for child in children])


def bm25_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    scores = bm25.get_scores(tokenize(question))
    ranked_indices = sorted(range(len(scores)), key=lambda index: scores[index], reverse=True)
    return [
        as_retrieved_document(children[index], float(scores[index]))
        for index in ranked_indices[:k]
    ]


bm25_preview = bm25_retrieve("What do BCS and MCS stand for?", k=4)
print_results(bm25_preview)


#1 | child-079 | page=13 | score=11.1440
could result in health problems. 83 No matter the life stage, to help avoid potential nutrient insufﬁciencies, cats should be fed diets labeled with an Association of American Feed Control Of ﬁcials statement of nutritional adequacy. AAHA and the AAFP do not a

#2 | child-029 | page=6 | score=10.8317
Evaluation and recording of body weight, body condition score (BCS), and muscle condition score (MCS) are important compo- nents of the physical examination at all life stages to allow early 56 JAAHA | 57:2 Mar/Apr 2021

#3 | child-006 | page=1 | score=9.7786
BCS (body condition score); DER (daily energy requirements); DJD (degenerative joint disease); FCV (feline calicivirus); FeLV (feline leukemia virus); FHV-1 (feline herpesvirus type 1); FIC (feline idiopathic cystitis); FPV (feline panleukopenia virus); GI (ga

#4 | child-082 | page=13 | score=9.7514
vidual patient. Recommendations can be found in the AAFP ’s Feline Feeding Programs Consensus S

### ❓ Question 2: When Should BM25 Win?

Which query is more likely to favor BM25: `What do BCS and MCS stand for?` or `How should an owner get ready for a senior-cat wellness visit?` Why?

### Answer

BM25 performs better when the query contains important exact keywords rather than relying on semantic similarity. It ranks documents based on keyword matching and word statistics such as term frequency and inverse document frequency (IDF).

BM25 favors the BCS/MCS query because it relies on exact keyword matching. Since these are rare domain-specific abbreviations, they receive a higher IDF weight and are more likely to rank the relevant document above others.

### 🚀 Activity 1: Dense vs. BM25 Failure Modes

Pick three questions: one paraphrase-heavy question, one exact-term question, and one broad multi-part question. Inspect both result lists before generating an answer.

- Which retriever put the best evidence first?

"What does the guideline say about BCS and MCS?": BM25
"How often should senior cats see a veterinarian?": Dense
"What factors shape an individualized preventive-care plan?": BM25 & Dense

- Which retriever returned redundant chunks?

None of the retrievers clearly returned redundant chunks based on the retrieved results. However, for the BCS/MCS query, the dense retriever returned neighboring chunks (child-005 and child-006) from the same page. Since overlapping chunking was used, these chunks may contain overlapping context and are the closest example of redundancy among the results.

- Which question type should favor BM25, and why?

one exact-term question should favor BM25 as BM25 relies on the exact keyword matching for ranking. one broad multi-part question may be considered to favor BM25 if it also contains important exact keywords or domain-specific terminology. 


In [11]:
activity_questions = [
    "What does the guideline say about BCS and MCS?",
    "How often should senior cats see a veterinarian?",
    "What factors shape an individualized preventive-care plan?",
]

for question in activity_questions:
    print(f"\nQUESTION: {question}\n")
    print("DENSE")
    print_results(dense_retrieve(question, k=3), text_limit=150)
    print("BM25")
    print_results(bm25_retrieve(question, k=3), text_limit=150)



QUESTION: What does the guideline say about BCS and MCS?

DENSE
#1 | child-005 | page=1 | score=0.4263
mendations should not be construed as dictating an exclusive protocol, course of treatment, or procedure. Variations in practice may be war- ranted ba

#2 | child-029 | page=6 | score=0.4119
Evaluation and recording of body weight, body condition score (BCS), and muscle condition score (MCS) are important compo- nents of the physical exami

#3 | child-006 | page=1 | score=0.3845
BCS (body condition score); DER (daily energy requirements); DJD (degenerative joint disease); FCV (feline calicivirus); FeLV (feline leukemia virus);

BM25
#1 | child-029 | page=6 | score=12.3015
Evaluation and recording of body weight, body condition score (BCS), and muscle condition score (MCS) are important compo- nents of the physical exami

#2 | child-082 | page=13 | score=11.5603
vidual patient. Recommendations can be found in the AAFP ’s Feline Feeding Programs Consensus Statement. 19 Young Adult Cats

## Task 5: Parent-Child Retrieval

Build a parent-child pipeline on top of the dense retriever:

question -> dense child search in Qdrant -> matching parent PDF pages

Child chunks give the vector store a focused search surface. Parent pages give the answer model surrounding context. Parent-child retrieval adds context recovery to dense retrieval; it is not hybrid retrieval.


In [12]:
# Parent-page lookup begins here; naive RAG does not need parent records.
parents_by_id = {page.metadata["page_id"]: page for page in pages}


def recover_parent_documents(
    child_candidates: Sequence[RetrievedDocument],
    k: int,
) -> list[RetrievedDocument]:
    parents: list[RetrievedDocument] = []
    seen_parent_ids: set[str] = set()

    for child in child_candidates:
        page_id = child.metadata["page_id"]
        if page_id in seen_parent_ids:
            continue
        parent = parents_by_id[page_id]
        parents.append(
            RetrievedDocument(
                id=page_id,
                text=parent.page_content,
                score=child.score,
                evidence_ids=(page_id,),
                metadata={
                    **parent.metadata,
                    "retrieved_from_child": child.id,
                    "first_stage_score": child.score,
                },
            )
        )
        seen_parent_ids.add(page_id)
        if len(parents) == k:
            break
    return parents


def parent_child_dense_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    child_candidates = dense_retrieve(question, k=FIRST_STAGE_K)
    return recover_parent_documents(child_candidates, k=k)


parent_preview = parent_child_dense_retrieve(
    "How should an owner prepare for a senior cat wellness visit?",
    k=3,
)
print_results(parent_preview, text_limit=400)


#1 | page-06 | page=6 | score=0.6861
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at the veterinarian ’s discretion. The guidelines are intended to be a starting point from which individualized care recommenda- tions can be developed. Discussion Items for All Life Stages The Task Force recommends a minimum of annual examinations for all ca

#2 | page-07 | page=7 | score=0.6556
detection of changes and identi ﬁcation of trends. 20 Obtaining dorsal and lateral photographs of the patient is recommended to facilitate monitoring BCS/MCS as the cat ages, and can help the owner recognize subtle changes. Diseases and conditions that require additional focus during the examination by each life stage are listed in Table 3. Kittens Kittens will have different health risks dependin

#3 | page-10 | page=10 | score=0.6434
including carpeting, window and door frames, curtains, and couches. Keeping the nail

### ❓ Question 3: Search Small, Return Large?

Why does parent-child retrieval search focused child chunks but return the larger parent pages instead of indexing and returning only full pages?

### Answer

Parent-child retrieval indexes smaller child chunks because their embeddings represent more focused topics, improving retrieval precision. Indexing only full pages would produce broader embeddings that may dilute the relevance of individual topics and make retrieval less accurate. After locating the relevant child chunk, the retriever returns the larger parent page so the LLM has enough surrounding context to generate a complete answer.


# Breakout Room #2: Combine, Rank, and Expand

The first half produced three building blocks: dense retrieval, BM25, and parent-child context recovery. This breakout combines them and compares the results.

In this breakout:

1. Fuse dense and BM25 rankings with reciprocal rank fusion.
2. Rerank the broad hybrid candidate set with Cohere.
3. Expand vague questions into multiple searches.
4. Compare the trade-offs with the local evaluation library.

Keep only the stages that improve measured retrieval enough to justify their latency.


## Task 6: Hybrid Retrieval — Dense + BM25

Dense and BM25 scores use different scales, so raw scores cannot be added directly. Reciprocal rank fusion (RRF) works from their **ranked lists** instead.

The hybrid first stage is:

dense candidates + BM25 candidates -> RRF -> hybrid child candidates

**Hybrid retrieval** combines semantic vector retrieval with sparse lexical retrieval. RRF makes it an **ensemble retriever** by fusing multiple rankings.


In [13]:
def reciprocal_rank_fusion(
    ranked_lists: Iterable[Sequence[RetrievedDocument]],
    *,
    limit: int,
    rrf_constant: int = 60,
) -> list[RetrievedDocument]:
    scores: dict[str, float] = {}
    documents_by_id: dict[str, RetrievedDocument] = {}

    for ranked_list in ranked_lists:
        for rank, document in enumerate(ranked_list, start=1):
            documents_by_id.setdefault(document.id, document)
            scores[document.id] = scores.get(document.id, 0.0) + 1 / (rrf_constant + rank)

    return [
        replace(
            documents_by_id[document_id],
            score=score,
            metadata={
                **documents_by_id[document_id].metadata,
                "rrf_score": score,
            },
        )
        for document_id, score in sorted(scores.items(), key=lambda item: item[1], reverse=True)[:limit]
    ]


def hybrid_children_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    return reciprocal_rank_fusion(
        [
            dense_retrieve(question, k=FIRST_STAGE_K),
            bm25_retrieve(question, k=FIRST_STAGE_K),
        ],
        limit=k,
    )


hybrid_preview = hybrid_children_retrieve("What does the guideline say about BCS and MCS?", k=4)
print_results(hybrid_preview)


#1 | child-029 | page=6 | score=0.0325
Evaluation and recording of body weight, body condition score (BCS), and muscle condition score (MCS) are important compo- nents of the physical examination at all life stages to allow early 56 JAAHA | 57:2 Mar/Apr 2021

#2 | child-030 | page=7 | score=0.0310
detection of changes and identi ﬁcation of trends. 20 Obtaining dorsal and lateral photographs of the patient is recommended to facilitate monitoring BCS/MCS as the cat ages, and can help the owner recognize subtle changes. Diseases and conditions that require

#3 | child-005 | page=1 | score=0.0164
mendations should not be construed as dictating an exclusive protocol, course of treatment, or procedure. Variations in practice may be war- ranted based on the needs of the individual patient, resources, and limitations unique to each individual practice sett

#4 | child-082 | page=13 | score=0.0161
vidual patient. Recommendations can be found in the AAFP ’s Feline Feeding Programs Consensus Stat

## Task 7: Cohere Reranking over Hybrid Candidates

Use hybrid retrieval to gather a broad set of plausible child chunks from dense and BM25 search. Recover their parent pages, then send those candidates and the question to Cohere Rerank.

dense + BM25 -> RRF -> parent pages -> Cohere Rerank -> final context

The result is a **two-stage hybrid retrieve-then-rerank pipeline**. Cohere scores rank documents within one query and candidate set; do not treat them as universal probabilities.

The custom RRF function supplies the candidate list, so the notebook calls LangChain's `CohereRerank` compressor directly. A single LangChain `BaseRetriever` could instead be wrapped with `ContextualCompressionRetriever`.


In [14]:
RERANK_CANDIDATE_K = 8


def to_langchain_document(candidate: RetrievedDocument) -> Document:
    return Document(
        page_content=candidate.text,
        metadata={
            **candidate.metadata,
            "retrieved_id": candidate.id,
            "evidence_ids": list(candidate.canonical_evidence_ids),
            "first_stage_score": candidate.score,
        },
    )


def to_reranked_document(document: Document) -> RetrievedDocument:
    relevance_score = document.metadata.get("relevance_score")
    return RetrievedDocument(
        id=document.metadata["retrieved_id"],
        text=document.page_content,
        score=float(relevance_score) if relevance_score is not None else None,
        evidence_ids=tuple(document.metadata["evidence_ids"]),
        metadata=dict(document.metadata),
    )


def rerank_parent_candidates(
    question: str, candidates: Sequence[RetrievedDocument], k: int
) -> list[RetrievedDocument]:
    compressor = CohereRerank(model=RERANK_MODEL, top_n=k)
    reranked_documents = compressor.compress_documents(
        documents=[to_langchain_document(candidate) for candidate in candidates],
        query=question,
    )
    return [to_reranked_document(document) for document in reranked_documents]


def hybrid_reranked_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    child_candidates = hybrid_children_retrieve(question, k=RERANK_CANDIDATE_K)
    parent_candidates = recover_parent_documents(child_candidates, k=RERANK_CANDIDATE_K)
    return rerank_parent_candidates(question, parent_candidates, k)


rerank_preview = hybrid_reranked_retrieve(
    "How should an owner prepare for a senior cat wellness visit?",
    k=3,
)
print_results(rerank_preview, text_limit=400)


#1 | page-06 | page=6 | score=0.7716
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at the veterinarian ’s discretion. The guidelines are intended to be a starting point from which individualized care recommenda- tions can be developed. Discussion Items for All Life Stages The Task Force recommends a minimum of annual examinations for all ca

#2 | page-18 | page=18 | score=0.7030
events to increase knowledge and con ﬁdence when taking patient histories (see “Conducting Effective Patient Histories” box) and pro- viding client education are just as important as further education on feline-friendly handling, disease processes, and technical skills. Ideally, client education is a key responsibility for all staff members. Every life stage will have speci ﬁc items that should be

#3 | page-10 | page=10 | score=0.6923
including carpeting, window and door frames, curtains, and couches. Keeping the nai

## Task 8: Multi-Query Retrieval

A user question may be vague, incomplete, or phrased differently from the source. Generate alternate search queries, run each through hybrid first-stage retrieval, and fuse the resulting child rankings.

Multi-query retrieval expands recall on top of the hybrid retrieve-then-rerank pipeline. It also adds model calls, latency, and possible noise.


In [15]:
multi_query_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Write {count} short, distinct search queries for the cat health guideline PDF. "
            "Return one query per line. Do not answer the question.",
        ),
        ("human", "{question}"),
    ]
)


def generate_query_variants(question: str, count: int = 3) -> list[str]:
    response = query_model.invoke(
        multi_query_prompt.format_messages(question=question, count=count)
    )
    variants = [question]
    for line in response.content.splitlines():
        candidate = re.sub(r"^\s*(?:[-*]|\d+[.)])\s*", "", line).strip()
        if candidate and candidate.lower() not in {item.lower() for item in variants}:
            variants.append(candidate)
    return variants[: count + 1]


def multi_query_hybrid_children(question: str, k: int = 5) -> list[RetrievedDocument]:
    ranked_lists: list[Sequence[RetrievedDocument]] = []
    for variant in generate_query_variants(question):
        ranked_lists.append(dense_retrieve(variant, k=FIRST_STAGE_K))
        ranked_lists.append(bm25_retrieve(variant, k=FIRST_STAGE_K))
    return reciprocal_rank_fusion(ranked_lists, limit=k)


def multi_query_reranked_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    child_candidates = multi_query_hybrid_children(question, k=RERANK_CANDIDATE_K)
    parent_candidates = recover_parent_documents(child_candidates, k=RERANK_CANDIDATE_K)
    return rerank_parent_candidates(question, parent_candidates, k)


question = "How should an owner prepare for a senior cat wellness visit?"
print(generate_query_variants(question))
print_results(multi_query_reranked_retrieve(question, k=4), text_limit=400)


['How should an owner prepare for a senior cat wellness visit?', 'senior cat wellness visit preparation PDF', 'cat owner prepare for senior cat checkup guideline', 'senior feline wellness appointment prep recommendations']
#1 | page-06 | page=6 | score=0.7716
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at the veterinarian ’s discretion. The guidelines are intended to be a starting point from which individualized care recommenda- tions can be developed. Discussion Items for All Life Stages The Task Force recommends a minimum of annual examinations for all ca

#2 | page-14 | page=14 | score=0.7030
good starting point is to calculate the adult feline patient ’s resting energy requirements (RER) according to the following calculation: RER (kcal per day) ¼ 30 × (body weight in kg) 1 70. Daily energy requirements (DER) are determined based on multiplying by a needs factor, which in the case of yo

## Task 9: Compare Retrieval and Answer Results

The local library measures retrieval and answer quality separately.

- **Retrieval:** Hit@k, Precision@k, Recall@k, MRR, and latency.
- **Faithfulness:** the share of answer statements supported by the passages retrieved for that answer. The OpenAI judge records a statement, reason, and 0/1 verdict for each claim.
- **Answer similarity:** cosine similarity between OpenAI embeddings of the generated answer and a reviewed reference answer.

First, compare five retrieval pipelines on the same reviewed text-retrieval cases:

1. Naive dense RAG.
2. BM25.
3. Dense parent-child retrieval.
4. Hybrid dense + BM25 retrieval with Cohere reranking.
5. Multi-query hybrid retrieve-then-rerank.

Inspect per-case rankings alongside aggregate metrics before choosing a system. The visual life-stage table needs separate handling: the PDF text extractor loses most of its cells.


In [64]:
text_reviewed_cases = [
    EvalCase(
        id="life-stage-definitions",
        query="What feline life stages does the guideline define?",
        relevant_evidence_ids=("page-03",),
        tags=("baseline", "life-stage"),
    ),
    EvalCase(
        id="senior-visit-frequency",
        query="How often should senior cats be seen by a veterinarian?",
        relevant_evidence_ids=("page-06",),
        tags=("exact", "senior"),
    ),
    EvalCase(
        id="bcs-mcs",
        query="What do BCS and MCS mean, and why are they recorded?",
        relevant_evidence_ids=("page-06", "page-07"),
        tags=("acronym", "dense-vs-bm25"),
    ),
]

table_layout_challenge = EvalCase(
    id="life-stage-table",
    query="How do wellness discussion items change across feline life stages?",
    relevant_evidence_ids=("page-04", "page-05"),
    tags=("table", "parent-child", "requires-visual-review"),
    notes="Use this after adding a PDF-page vision fallback.",
)

EVAL_K = 4
dense_report = run_retrieval_eval("naive_dense", text_reviewed_cases, dense_retrieve, k=EVAL_K)
bm25_report = run_retrieval_eval("bm25", text_reviewed_cases, bm25_retrieve, k=EVAL_K)
parent_child_report = run_retrieval_eval(
    "dense_parent_child",
    text_reviewed_cases,
    parent_child_dense_retrieve,
    k=EVAL_K,
)
hybrid_rerank_report = run_retrieval_eval(
    "hybrid_rrf_then_cohere",
    text_reviewed_cases,
    hybrid_reranked_retrieve,
    k=EVAL_K,
)
multi_query_report = run_retrieval_eval(
    "multi_query_hybrid_rerank",
    text_reviewed_cases,
    multi_query_reranked_retrieve,
    k=EVAL_K,
)

comparison = compare_reports(
    dense_report,
    bm25_report,
    parent_child_report,
    hybrid_rerank_report,
    multi_query_report,
)
comparison


,retriever,k,cases,hit_rate,precision_at_k,recall_at_k,mrr,mean_latency_ms
0,dense_parent_child,4,3,1.0,0.333333,1.0,1.000000,522.431152
1,naive_dense,4,3,1.0,0.333333,1.0,1.000000,620.293375
2,hybrid_rrf_then_cohere,4,3,1.0,0.333333,1.0,0.777778,490.895514
3,multi_query_hybrid_rerank,4,3,1.0,0.333333,1.0,0.750000,2230.597958
4,bm25,4,3,1.0,0.333333,1.0,0.583333,0.310986


In [65]:
parent_child_report.case_table()

,case_id,query,tags,retrieved_ids,hit@k,precision@k,recall@k,reciprocal_rank,latency_ms
0,life-stage-definitions,What feline life stages does the guideline def...,"baseline, life-stage","[page-03, page-07, page-02, page-01]",1.0,0.25,1.0,1.0,760.208583
1,senior-visit-frequency,How often should senior cats be seen by a vete...,"exact, senior","[page-06, page-16, page-07, page-10]",1.0,0.25,1.0,1.0,599.300916
2,bcs-mcs,"What do BCS and MCS mean, and why are they rec...","acronym, dense-vs-bm25","[page-06, page-01, page-07, page-19]",1.0,0.50,1.0,1.0,207.783958


### Answer-Level Evaluation: Faithfulness and Answer Similarity

Retrieval success does not guarantee that an answer stays grounded in its retrieved passages or covers the reviewed answer. The cases below add reviewed reference answers.

The generic runner follows the same shape as Evalite: data, task, and scorers. This comparison uses the naive dense baseline and the complete multi-query pipeline. Each answer is scored against its own retrieved passages for faithfulness, then against the same reviewed reference answer for semantic similarity.


In [66]:
answer_reviewed_cases = [
    EvalItem(
        id="life-stage-definitions",
        input="What feline life stages does the guideline define?",
        expected=(
            "The guidelines define kitten (birth to 1 year), young adult (1 through 6 years), "
            "mature adult (7 through 10 years), and senior (over 10 years). End of life can occur at any age."
        ),
        tags=("life-stage",),
    ),
    EvalItem(
        id="senior-visit-frequency",
        input="How often should senior cats be seen by a veterinarian?",
        expected=(
            "All cats should have at least annual examinations. Senior cats should be seen at least every "
            "6 months, with more frequent visits for chronic conditions."
        ),
        tags=("senior",),
    ),
    EvalItem(
        id="bcs-mcs",
        input="What do BCS and MCS mean, and why are they recorded?",
        expected=(
            "BCS is body condition score and MCS is muscle condition score. Along with body weight, "
            "they should be evaluated and recorded at every life stage to identify changes and trends early."
        ),
        tags=("acronym",),
    ),
]

faithfulness_judge = make_openai_faithfulness_judge(evaluation_model)
answer_scorers = [
    faithfulness_scorer(faithfulness_judge),
    answer_similarity_scorer(embeddings),
]


def answerer_for(retriever):
    return lambda question: answer_with_retriever(question, retriever)

In [70]:
dense_answer_report = run_eval(
    "dense_answer", 
    data=answer_reviewed_cases,
    task=answerer_for(dense_retrieve),
    scorers=answer_scorers,
)
bm25_answer_report = run_eval(
    "bm25_answer", 
    data=answer_reviewed_cases,
    task=answerer_for(bm25_retrieve),
    scorers=answer_scorers,
)
parent_child_answer_report = run_eval(
    "dense_parent_child_answer",
    data=answer_reviewed_cases,
    task=answerer_for(parent_child_dense_retrieve),
    scorers=answer_scorers,
)
hybrid_rerank_answer_report = run_eval(
    "hybrid_rrf_then_cohere_answer",
    data=answer_reviewed_cases,
    task=answerer_for(hybrid_reranked_retrieve),
    scorers=answer_scorers,
)
full_pipeline_answer_report = run_eval(
    "multi_query_hybrid_rerank_answer",
    data=answer_reviewed_cases,
    task=answerer_for(multi_query_reranked_retrieve),
    scorers=answer_scorers,
)

answer_comparison = compare_eval_reports(
    dense_answer_report,
    bm25_answer_report,
    parent_child_answer_report,
    hybrid_rerank_answer_report,
    full_pipeline_answer_report,
)

answer_comparison

,evaluation,cases,faithfulness,answer_similarity,mean_task_latency_ms,mean_scoring_latency_ms
0,dense_parent_child_answer,3,1.000000,0.861150,1492.158528,2050.365417
1,dense_answer,3,1.000000,0.855706,1431.168708,2059.599222
2,multi_query_hybrid_rerank_answer,3,0.933333,0.863376,3322.352389,4197.839861
3,hybrid_rrf_then_cohere_answer,3,0.866667,0.867455,1775.503889,2198.653195
4,bm25_answer,3,0.833333,0.808507,1190.887500,2022.775417


In [71]:
parent_child_answer_report.case_table()

,case_id,input,expected,tags,output,task_latency_ms,scoring_latency_ms,faithfulness,faithfulness_metadata,answer_similarity,answer_similarity_metadata
0,life-stage-definitions,What feline life stages does the guideline def...,The guidelines define kitten (birth to 1 year)...,life-stage,AnswerOutput(answer='The guideline defines fiv...,1341.137208,2513.842084,1.0,{'verdicts': [{'statement': 'The guideline def...,0.850113,{'raw_cosine_similarity': 0.8501133942417766}
1,senior-visit-frequency,How often should senior cats be seen by a vete...,All cats should have at least annual examinati...,senior,AnswerOutput(answer='Senior cats should be see...,1391.251500,1766.690875,1.0,{'verdicts': [{'statement': 'Senior cats shoul...,0.845134,{'raw_cosine_similarity': 0.8451342077455084}
2,bcs-mcs,"What do BCS and MCS mean, and why are they rec...",BCS is body condition score and MCS is muscle ...,acronym,AnswerOutput(answer='BCS means **body conditio...,1744.086875,1870.563291,1.0,{'verdicts': [{'statement': 'BCS means body co...,0.888202,{'raw_cosine_similarity': 0.8882021027526252}


In [72]:
full_pipeline_answer_report.case_table()

,case_id,input,expected,tags,output,task_latency_ms,scoring_latency_ms,faithfulness,faithfulness_metadata,answer_similarity,answer_similarity_metadata
0,life-stage-definitions,What feline life stages does the guideline def...,The guidelines define kitten (birth to 1 year)...,life-stage,AnswerOutput(answer='The guideline defines fiv...,3546.123833,8564.059792,1.0,{'verdicts': [{'statement': 'The guideline def...,0.865576,{'raw_cosine_similarity': 0.8655756064154957}
1,senior-visit-frequency,How often should senior cats be seen by a vete...,All cats should have at least annual examinati...,senior,AnswerOutput(answer='Senior cats should be see...,3310.249083,1389.461750,1.0,{'verdicts': [{'statement': 'Senior cats shoul...,0.826736,{'raw_cosine_similarity': 0.8267357488010704}
2,bcs-mcs,"What do BCS and MCS mean, and why are they rec...",BCS is body condition score and MCS is muscle ...,acronym,AnswerOutput(answer='BCS means **body conditio...,3110.684250,2639.998042,0.8,{'verdicts': [{'statement': 'BCS means body co...,0.897817,{'raw_cosine_similarity': 0.8978167704431774}


### ❓ Question 4: Is More Retrieval Always Better?

Suppose multi-query plus reranking improves recall but lowers MRR and adds noticeable latency. How would faithfulness and answer similarity change your decision about shipping it for every cat-health question?

### Answer

Retrieval with good recall but bad MRR means it did retrieve most of the relevant documents but the relevant ones are not considered as the most important ones, these are intermediate results fed into the model for better context. However, noticeable latency definitely can harm user experiences if it's applied to all questions. Faithfulness and answer similarity help evaluate whether the final answer is grounded in the retrieved evidence and semantically close to the reviewed reference answer.

If both the faithfulness and answer similarity get improved enough to justify the additional latency, I would consider deploying it, especially for broader and more complex questions. Otherwise, I would avoid enabling it for every query and reserve it only for queries that the quality gain outweighs the latency cost.

### 🚀 Activity 2: Make and Defend a Retrieval Recommendation

Choose one retrieval result and one answer-evaluation result, then make a recommendation for the cat-health application.

Include:

1. Which rung of the retrieval ladder you chose.
2. Evidence from at least two metrics and one inspected ranking or claim-level verdict.
3. Its cost/latency trade-off.
4. One case where you would choose a different rung instead.

A higher aggregate metric does not settle the product decision.

I would recommend parent-child retrieval as default for the cat-health application because it retrieves the correct evidence for all reviewed text cases (recall@k=1.0), ranks relevant evidence highly(MRR=1.0) and the further generated answer can be fully supported by the evidence retrieved (faithfulness=1.0). The answer_similarity is pretty good with so much better task completion latency than the other two that are slightly higher than it.  

For bcs-mcs case, parent-child retrieval retrieved all the relevant documents referenced (page-06 & page-07) and the generate answer also highly matches the referenced answer with answer_similarity=0.888202. Faithfulness=1.0 means the final answer from model is fully supported by the evidence found.

It did show slightly higher task latency when comparing against other simpler retrieval methods like bm25 & naive dense. Also with the full content retrieved from the parent document passed into model for bigger context window, which would definitely increase the token cost here.

However, for questions that are much vague & broader, require evidence from multiple sections and where answer quality is more important than latency, I would choose multi-query rerank retrieval instead.

## Task 10: Answer with a Selected Pipeline

Pass only the selected pipeline's retrieved context to the answer model. Source labels keep the evidence inspectable. For the two-page life-stage table, inspect the original PDF page when text extraction does not preserve the table structure.


In [49]:
result = answer_with_retriever(
    "What should an owner discuss at a senior cat wellness visit?",
    retriever=parent_child_dense_retrieve,
)
print(result.answer)
print("\nRetrieved evidence:")
print_results(result.documents, text_limit=200)


At a senior cat wellness visit, the owner should discuss:

- Changes in appetite
- Increased drinking and urination
- Vomiting, vomiting hairballs, or diarrhea
- Increased nocturnal activity and vocalization
- Changes in the cat’s normal habits or activity
- Changes in litter box usage
- Changes in jumping or climbing
- Any signs of pain, reduced mobility, or reduced vision
- Changes in grooming habits
- Evaluation of quality of life

The guidelines also note that senior cats exhibiting new or unusual behavior should be evaluated for medical conditions, but I can’t go beyond what the context states.

Sources: page-07, page-08, page-10, page-06

Retrieved evidence:
#1 | page-07 | page=7 | score=0.6958
detection of changes and identi ﬁcation of trends. 20 Obtaining dorsal and lateral photographs of the patient is recommended to facilitate monitoring BCS/MCS as the cat ages, and can help the owner re

#2 | page-06 | page=6 | score=0.6693
For example, some senior cats aged 10 years and old

## Advanced Builds

- Add maximal marginal relevance (MMR) and measure whether it reduces redundant chunks.
- Add HyDE: generate a hypothetical answer, embed it, and use it as an alternate dense query.
- Add a PDF-page vision fallback for the life-stage table challenge.
- Add a second reviewed corpus and use metadata routing to avoid mixing evidence sets.

## Recap

Build in this order:

1. Start with a transparent naive dense RAG baseline in in-memory Qdrant.
2. Add BM25 to see the value of lexical retrieval.
3. Add parent-child recovery to improve answer context.
4. Combine dense and BM25 with RRF: hybrid / ensemble retrieval.
5. Rerank the hybrid parent candidates with Cohere: a two-stage retrieve-then-rerank pipeline.
6. Add multi-query expansion only when its recall benefit justifies its extra latency and cost.

Use the local evaluation library to inspect the retrieved evidence behind every aggregate number.
